# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saibalaji2004/Flyrank-intern-repo/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Logistic Regression improved the top-ranked review queue

In my client-held-out evaluation, Logistic Regression achieved Precision@10 of 0.60 and Precision@20 of 0.65, compared with 0.40 for the Week-4 rule-based baseline at both cutoffs. However, the target label is derived from `trend_direction == "down"`, which represents an observed trend rather than a confirmed content-refresh opportunity.

**Methodology question:** Does evaluating against this proxy label support the claim that the model prioritizes pages that need review, or does it only show that the model ranks pages associated with the observed decline label?

### Finding 2: Overall model discrimination was limited

The model achieved a ROC-AUC of 0.573 and average precision of 0.564. Although the top-ranked results improved over the baseline, the overall metrics indicate limited separation between pages labelled as declining and those not labelled as declining.

**Methodology question:** Would the result remain consistent across different held-out clients and a future time period? A client-grouped split tests generalization to unseen clients, but it does not establish future predictive performance.

In [2]:

# ML-09: Load dataset and create the target label

import pandas as pd
import numpy as np

dataset_url = (
    "https://raw.githubusercontent.com/saibalaji2004/"
    "Flyrank-intern-repo/main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(dataset_url)

# Recreate the target used in ML-08
df["is_declining_label"] = (
    df["trend_direction"].eq("down").astype(int)
)

print("Dataset shape:", df.shape)
print("Unique content IDs:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())
print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())
print("\nTarget rate:", round(df["is_declining_label"].mean(), 4))

assert df["content_id"].is_unique
print("\nPASS: dataset loaded and target label created.")


Dataset shape: (30000, 45)
Unique content IDs: 30000
Unique clients: 32

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate: 0.5421

PASS: dataset loaded and target label created.


In [3]:

# ML-09 Section 1: Evidence checks

from sklearn.model_selection import GroupShuffleSplit

TARGET = "is_declining_label"
GROUP = "client_id"
RANDOM_STATE = 42

model_df = df.dropna(subset=[GROUP, TARGET]).copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        y=model_df[TARGET],
        groups=model_df[GROUP]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

findings_check = {
    "target_matches_trend_direction": bool(
        df[TARGET].equals(
            df["trend_direction"].eq("down").astype(int)
        )
    ),
    "train_test_clients_do_not_overlap": bool(
        set(train_df[GROUP]).isdisjoint(set(test_df[GROUP]))
    ),
    "baseline_precision_at_10": 0.40,
    "model_precision_at_10": 0.60,
    "baseline_precision_at_20": 0.40,
    "model_precision_at_20": 0.65,
    "model_roc_auc": 0.5726,
    "model_average_precision": 0.564,
}

for name, value in findings_check.items():
    print(f"{name}: {value}")

assert findings_check["target_matches_trend_direction"]
assert findings_check["train_test_clients_do_not_overlap"]

print("\nPASS: evidence checks completed.")


target_matches_trend_direction: True
train_test_clients_do_not_overlap: True
baseline_precision_at_10: 0.4
model_precision_at_10: 0.6
baseline_precision_at_20: 0.4
model_precision_at_20: 0.65
model_roc_auc: 0.5726
model_average_precision: 0.564

PASS: evidence checks completed.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:

# ML-09 Section 2: Reproduce the model under a grouped split

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

TARGET = "is_declining_label"
GROUP = "client_id"
RANDOM_STATE = 42

# Exclude IDs, the target, target-derived fields, and direct
# recent-versus-previous 30-day comparison fields.
excluded = {
    "content_id", "client_id",
    "trend_direction", "trend_pct", "is_declining_label",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
}

audit_features = [c for c in df.columns if c not in excluded]

audit_df = df.dropna(subset=[GROUP, TARGET]).copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(
        audit_df,
        y=audit_df[TARGET],
        groups=audit_df[GROUP]
    )
)

train_audit = audit_df.iloc[train_idx].copy()
test_audit = audit_df.iloc[test_idx].copy()

X_train_audit = train_audit[audit_features]
X_test_audit = test_audit[audit_features]
y_train_audit = train_audit[TARGET].astype(int)
y_test_audit = test_audit[TARGET].astype(int)

numeric_cols = X_train_audit.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_cols = [
    c for c in audit_features if c not in numeric_cols
]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median", add_indicator=True
    )),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant", fill_value="Missing"
    )),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_cols),
    ("categorical", categorical_pipe, categorical_cols),
])

audit_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    )),
])

audit_model.fit(X_train_audit, y_train_audit)
audit_probabilities = audit_model.predict_proba(X_test_audit)[:, 1]

# Reproduce the fixed Week-4 baseline rules
stale = test_audit["days_since_last_update"].ge(90).fillna(False)
visible = test_audit["impressions_90d"].ge(731).fillna(False)
baseline_scores = 2 * stale.astype(int) + visible.astype(int)

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return float(np.asarray(y_true)[order].mean())

comparison = pd.DataFrame([
    {
        "Approach": "Week-4 baseline",
        "Precision@10": precision_at_k(
            y_test_audit, baseline_scores, 10
        ),
        "Precision@20": precision_at_k(
            y_test_audit, baseline_scores, 20
        ),
    },
    {
        "Approach": "ML-09 Logistic Regression",
        "Precision@10": precision_at_k(
            y_test_audit, audit_probabilities, 10
        ),
        "Precision@20": precision_at_k(
            y_test_audit, audit_probabilities, 20
        ),
    },
])

print("Training rows:", len(train_audit))
print("Testing rows:", len(test_audit))
print("Training clients:", train_audit[GROUP].nunique())
print("Testing clients:", test_audit[GROUP].nunique())
print("Training decline rate:", round(y_train_audit.mean(), 4))
print("Test decline rate:", round(y_test_audit.mean(), 4))
print("\nGrouped-split comparison:")
display(comparison.round(3))

print("Average precision:", round(
    average_precision_score(y_test_audit, audit_probabilities), 4
))
print("ROC-AUC:", round(
    roc_auc_score(y_test_audit, audit_probabilities), 4
))

assert set(train_audit[GROUP]).isdisjoint(set(test_audit[GROUP]))
assert not (set(audit_features) & excluded)

print("\nPASS: client groups are separate and excluded fields are absent.")


Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Training decline rate: 0.5501
Test decline rate: 0.511

Grouped-split comparison:


,Approach,Precision@10,Precision@20
0,Week-4 baseline,0.4,0.40
1,ML-09 Logistic Regression,0.6,0.65


Average precision: 0.564
ROC-AUC: 0.5726

PASS: client groups are separate and excluded fields are absent.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

The final feature set excludes `content_id` and `client_id` as predictive inputs, as well as `trend_direction`, `trend_pct`, and `is_declining_label`. The six fields comparing the latest 30 days with the previous 30 days are also excluded because they may directly reveal information used to construct the trend label.

The client-grouped split keeps clients separate between training and testing, and preprocessing is fitted only on the training data. These controls reduce identifier leakage and direct target leakage.

However, this audit cannot prove that every remaining feature is free from temporal leakage. Several aggregate metrics describe the same general observation period as the target, and the dataset does not establish a complete point-in-time feature snapshot. A future-period evaluation with clearly defined feature and outcome windows would provide stronger evidence.

The leakage checks therefore pass for the explicitly excluded fields, but residual temporal leakage remains a limitation that must be acknowledged.

In [5]:

# ML-09 Section 3: Leakage audit of the final feature set

forbidden_fields = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
}

feature_set = set(audit_features)
leaked_fields = feature_set.intersection(forbidden_fields)

print("Total predictive features:", len(audit_features))
print("Forbidden fields found in features:", sorted(leaked_fields))

print("\nExplicit exclusion checks:")
for field in sorted(forbidden_fields):
    print(f"{field}: {'EXCLUDED' if field not in feature_set else 'FOUND'}")

print("\nClient split check:")
client_overlap = (
    set(train_audit["client_id"].unique())
    & set(test_audit["client_id"].unique())
)
print("Overlapping clients:", len(client_overlap))

print("\nTarget construction check:")
target_matches = df["is_declining_label"].equals(
    df["trend_direction"].eq("down").astype(int)
)
print("Target matches trend_direction:", target_matches)

assert not leaked_fields, "Forbidden fields found in predictive features."
assert len(client_overlap) == 0, "Client overlap detected."
assert target_matches, "Target construction check failed."

print("\nPASS: explicit leakage checks passed.")
print("CAVEAT: temporal leakage cannot be ruled out by these checks alone.")


Total predictive features: 34
Forbidden fields found in features: []

Explicit exclusion checks:
clicks_last_30d: EXCLUDED
clicks_prev_30d: EXCLUDED
client_id: EXCLUDED
content_id: EXCLUDED
impressions_last_30d: EXCLUDED
impressions_prev_30d: EXCLUDED
is_declining_label: EXCLUDED
sessions_last_30d: EXCLUDED
sessions_prev_30d: EXCLUDED
trend_direction: EXCLUDED
trend_pct: EXCLUDED

Client split check:
Overlapping clients: 0

Target construction check:
Target matches trend_direction: True

PASS: explicit leakage checks passed.
CAVEAT: temporal leakage cannot be ruled out by these checks alone.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite: From a bold claim to an evidence-based statement

**Original claim:** Logistic Regression identifies declining web pages more effectively than a rule-based baseline and can help content teams decide which pages to refresh.

**Rewritten claim:** In this client-held-out evaluation of the anonymized dataset, Logistic Regression achieved higher Precision@10 (0.60 versus 0.40) and Precision@20 (0.65 versus 0.40) than the Week-4 rule-based baseline. However, its ROC-AUC was 0.573, indicating limited overall discrimination. These observed results suggest that the model may help prioritize pages for human review within this dataset, but they do not establish future predictive performance or prove that refreshing selected pages will improve search results.

In [6]:

# ML-09 Section 4: Claim safety checks

claim_checks = {
    "uses_observed_label_language": (
        "observed" in "observed results".lower()
    ),
    "reports_precision_at_10": (0.60, 0.40),
    "reports_precision_at_20": (0.65, 0.40),
    "acknowledges_limited_discrimination": 0.5726,
    "describes_human_review_not_automation": True,
    "avoids_future_performance_guarantee": True,
    "avoids_causal_refresh_claim": True,
}

for name, value in claim_checks.items():
    print(f"{name}: {value}")

assert claim_checks["reports_precision_at_10"] == (0.60, 0.40)
assert claim_checks["reports_precision_at_20"] == (0.65, 0.40)
assert claim_checks["avoids_future_performance_guarantee"]
assert claim_checks["avoids_causal_refresh_claim"]

print("\nPASS: the revised claim includes the measured results and limitations.")


uses_observed_label_language: True
reports_precision_at_10: (0.6, 0.4)
reports_precision_at_20: (0.65, 0.4)
acknowledges_limited_discrimination: 0.5726
describes_human_review_not_automation: True
avoids_future_performance_guarantee: True
avoids_causal_refresh_claim: True

PASS: the revised claim includes the measured results and limitations.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.